In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [15]:
X = torch.rand(1,28,28, device=device)
logits = model(X)
pred_prob = nn.Softmax(dim=1)(logits)
y_pred = pred_prob.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4], device='cuda:0')


In [16]:
# taking a sample of 3 images => (3, 28 * 28)
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [17]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.shape)

torch.Size([3, 784])


In [25]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [26]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2931, -0.3963,  0.2468, -0.4875,  0.3690, -0.5789,  0.4768, -0.6448,
         -0.5651, -0.4092, -0.2935,  0.5386,  0.3451,  0.1139,  0.0995, -0.1334,
         -0.2941,  0.0760, -0.1330, -0.0693],
        [-0.0637, -0.1284,  0.0672, -0.5977,  0.4673, -0.7804,  0.1819, -0.6068,
         -0.6936, -0.2807, -0.1870,  0.3680,  0.4671,  0.2376,  0.4071,  0.0167,
         -0.2613, -0.0615, -0.0405,  0.3357],
        [ 0.2541, -0.3486,  0.1808, -0.4530,  0.6155, -0.7151,  0.1659, -0.3824,
         -0.4700, -0.0477, -0.2284,  0.3761,  0.4303,  0.2815,  0.5478, -0.0138,
         -0.1195,  0.0388, -0.3894, -0.0143]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2931, 0.0000, 0.2468, 0.0000, 0.3690, 0.0000, 0.4768, 0.0000, 0.0000,
         0.0000, 0.0000, 0.5386, 0.3451, 0.1139, 0.0995, 0.0000, 0.0000, 0.0760,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0672, 0.0000, 0.4673, 0.0000, 0.1819, 0.0000, 0.0000,
         0.0000, 0.0000, 0.3680, 0.4671, 0.2376, 0.40

In [28]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
print(f"{seq_modules}\n\n")
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=20, bias=True)
  (2): ReLU()
  (3): Linear(in_features=20, out_features=10, bias=True)
)


tensor([[-0.0850, -0.1192, -0.1523, -0.0933, -0.0029, -0.1051, -0.0322, -0.2933,
          0.1731,  0.1138],
        [-0.1826, -0.1333, -0.2142, -0.0728,  0.0273,  0.0289, -0.2173, -0.2349,
          0.1174,  0.0047],
        [-0.1736, -0.0395, -0.2965, -0.0205, -0.0716, -0.1743, -0.2037, -0.1973,
          0.1547,  0.0904]], grad_fn=<AddmmBackward0>)


In [32]:
softmax = nn.Softmax(dim=1)
pred_prob = softmax(logits)
print(pred_prob)

tensor([[0.0967, 0.0935, 0.0904, 0.0959, 0.1050, 0.0948, 0.1020, 0.0785, 0.1252,
         0.1180],
        [0.0903, 0.0949, 0.0875, 0.1008, 0.1114, 0.1116, 0.0872, 0.0857, 0.1219,
         0.1089],
        [0.0914, 0.1045, 0.0809, 0.1066, 0.1012, 0.0914, 0.0887, 0.0893, 0.1270,
         0.1190]], grad_fn=<SoftmaxBackward0>)


In [41]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0049,  0.0093,  0.0268,  ..., -0.0256, -0.0132,  0.0286],
        [-0.0167, -0.0030,  0.0031,  ..., -0.0165, -0.0140, -0.0258]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0214, -0.0355], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[-0.0419, -0.0432, -0.0041,  ..., -0.0260, -0.0042,  0.0420],
        [ 0.0167,  0.0188,  0.0123,  ...,  0.0433, -0.0042,  0.0264]],
       device='cuda:0', grad_fn=<Slice